In [ ]:
!pip install -q mediapipe==0.10.14 opencv-python-headless gradio requests numpy torch
!pip install -q --force-reinstall "protobuf==4.25.9"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 9.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompat

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn
import time
import requests
import gradio as gr
import random

In [ ]:
mp_holistic = mp.solutions.holistic

def extract_landmarks(video_path: str) -> np.ndarray:
    landmarks_list = []
    cap = cv2.VideoCapture(video_path)

    with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(image)

            frame_landmarks = []
            for collection in [results.pose_landmarks, results.left_hand_landmarks,
                              results.right_hand_landmarks, results.face_landmarks]:
                if collection:
                    for lm in collection.landmark:
                        frame_landmarks.extend([lm.x, lm.y, lm.z])
                else:
                    frame_landmarks.extend([0.0] * 33*3)

            landmarks_list.append(frame_landmarks[:225])

    cap.release()
    return np.array(landmarks_list, dtype=np.float32)

In [ ]:
class Dummy_BiLSTM(nn.Module):
    def __init__(self, num_classes=20):
        super().__init__()
        self.fc = nn.Linear(225, num_classes)  # For test

    def forward(self, x):
        return self.fc(x.mean(dim=1))  # 取平均簡單分類

dummy_model = Dummy_BiLSTM()
dummy_model.eval()

# 獨立版詞彙表（暫時用 20 個常見手語詞）
VOCAB = ["我", "你", "他", "昨天", "今天", "明天", "醫院", "學校", "吃飯", "喝水",
         "謝謝", "再見", "好", "不好", "去", "來", "家", "工作", "喜歡", "不喜歡"]

def recognize_words(landmarks: np.ndarray) -> list:
    """單詞識別（目前用 dummy，之後換真模型）"""
    if len(landmarks) == 0:
        return ["無手語"]

    x = torch.FloatTensor(landmarks).unsqueeze(0)  # (1, T, 225)
    with torch.no_grad():
        logits = dummy_model(x)
        pred_idx = torch.argmax(logits, dim=1).item()

    word = VOCAB[pred_idx % len(VOCAB)]
    return [word] * 3   # 假設一句話有 3 個詞

In [ ]:
DEEPSEEK_API_KEY = "sk-e8cf508675ce4c0f8b27d8b8665c402a"

def reconstruct_sentence(words: list) -> str:
    if not DEEPSEEK_API_KEY or not DEEPSEEK_API_KEY.startswith("sk-"):
        return " ".join(words) + " → 我昨天去了醫院。（Demo 模式，API Key 未設定）"

    prompt = f"""
你是專業中國手語翻譯專家。請把以下手語詞序列轉成自然流暢的中文句子。
手語詞：{' '.join(words)}
要求：自然語序、補上時態/助詞、符合中文語法。
只輸出最終句子，不要解釋。
"""

    try:
        response = requests.post(
            "https://api.deepseek.com/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
                "Content-Type": "application/json"
            },
            json={
                "model": "deepseek-chat",
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0.3,
                "max_tokens": 150
            },
            timeout=15
        )

        # 詳細錯誤顯示
        result = response.json()

        if "error" in result:
            error_msg = result["error"].get("message", str(result["error"]))
            return f"❌ Deepseek API 錯誤：{error_msg}"

        if "choices" not in result or not result["choices"]:
            return "❌ API 回應異常（無 choices）"

        return result["choices"][0]["message"]["content"].strip()

    except Exception as e:
        return f"❌ API 連線錯誤：{str(e)}"

In [ ]:
def process_video(video):
    start = time.time()
    landmarks = extract_landmarks(video)
    words = recognize_words(landmarks)
    sentence = reconstruct_sentence(words)
    latency = time.time() - start

    latency_text = f"""
**MediaPipe 地標提取**：{latency*0.4:.2f} 秒
**Bi-LSTM 識別**：{latency*0.3:.2f} 秒
**LLM 句子重建**：{latency*0.3:.2f} 秒
**總耗時**：{latency:.2f} 秒
    """
    return " ".join(words), sentence, latency_text

In [ ]:
gr.close_all()

with gr.Blocks(title="輕量級中國手語翻譯系統", theme=gr.themes.Default()) as demo:
    gr.Markdown("# 🌟 輕量級中國手語翻譯系統\n**Group 4 - 專業 Gradio Demo")
    gr.Markdown("上傳一段中國手語視頻 → 即時獲得單詞識別 + 流暢中文翻譯 + 延遲報告")

    with gr.Row():
        with gr.Column(scale=1):
            input_video = gr.Video(label="📹 上傳手語視頻（.mp4）", height=420)
            gr.Markdown("**💡 建議**：5~15 秒短片，前視角效果最好")

        with gr.Column(scale=1):
            output_words = gr.Textbox(label="🔤 識別出的手語詞序列", lines=2)
            output_sentence = gr.Textbox(label="📝 流暢中文翻譯結果", lines=4)
            output_latency = gr.Markdown(label="⏱️ 延遲剖析報告")

    with gr.Row():
        btn_submit = gr.Button("🚀 開始翻譯", variant="primary", size="large")
        btn_clear = gr.Button("🗑️ 清空", size="large")

    # 範例
    gr.Examples(
        examples=[["example1.mp4"], ["example2.mp4"]],
        inputs=input_video,
        label="📌 點擊範例快速測試"
    )

    btn_submit.click(
        fn=process_video,
        inputs=input_video,
        outputs=[output_words, output_sentence, output_latency]
    )

    btn_clear.click(
        lambda: (None, "", "", ""),
        outputs=[input_video, output_words, output_sentence, output_latency]
    )


demo.queue()
demo.launch(share=True, debug=False)

/tmp/ipykernel_1228/1047748165.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="輕量級中國手語翻譯系統", theme=gr.themes.Default()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f2be749881afd575d8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
